In [1]:
import pandas as pd

In [2]:
# Lista de arquivos da secretaria
arquivos = [
    'dados_secretaria/dados2021.csv',
    'dados_secretaria/dados2022.csv',
    'dados_secretaria/dados2023.csv',
    'dados_secretaria/dados2024.csv',
    'dados_secretaria/dados2025.csv',
    'dados_secretaria/dados2026.csv',
]

dfs = []

# Leitura em chunks para evitar estouro de RAM
for arq in arquivos:
    for chunk in pd.read_csv(arq, sep=';', encoding='latin1', chunksize=50000):

        # Padronizar nomes das colunas
        chunk.columns = chunk.columns.str.lower().str.strip()

        # Remover colunas inúteis (ex: Unnamed)
        chunk = chunk.loc[:, ~chunk.columns.str.contains('^unnamed')]
        chunk = chunk.loc[:, chunk.columns != '...']

        # Converter coluna de data
        chunk = chunk.dropna(subset=['data fato'])
        chunk['data fato'] = pd.to_datetime(chunk['data fato'], errors='coerce', dayfirst=True)
        chunk = chunk.dropna(subset=['data fato'])

        # Padronizar colunas de texto
        for col in ['tipo fato', 'grupo fato', 'municipio fato', 'bairro', 'local fato', 'tipo enquadramento']:
            if col in chunk.columns:
                chunk[col] = chunk[col].astype(str).str.lower().str.strip()

        dfs.append(chunk)

# Unificar todos os chunks
df_crimes = pd.concat(dfs, ignore_index=True)

# Remover duplicatas
df_crimes = df_crimes.drop_duplicates()

C:\Users\sinpl\AppData\Local\Temp\ipykernel_12848\31202489.py:15: DtypeWarning: Columns (0: Unnamed: 23, 1: Unnamed: 24, 2: Unnamed: 26, 3: Unnamed: 27, 4: Unnamed: 29, 5: Unnamed: 30, 6: Unnamed: 32, 7: Unnamed: 33, 8: Unnamed: 35, 9: Unnamed: 36, 10: Unnamed: 38, 11: Unnamed: 39, 12: Unnamed: 41, 13: Unnamed: 42, 14: Unnamed: 44, 15: Unnamed: 45, 16: Unnamed: 47, 17: Unnamed: 48, 18: Unnamed: 50, 19: Unnamed: 51, 20: Unnamed: 53, 21: Unnamed: 54, 22: Unnamed: 56, 23: Unnamed: 57, 24: Unnamed: 59, 25: Unnamed: 60, 26: Unnamed: 62, 27: Unnamed: 63, 28: Unnamed: 65, 29: Unnamed: 66, 30: Unnamed: 68, 31: Unnamed: 69, 32: Unnamed: 71, 33: Unnamed: 72, 34: Unnamed: 74, 35: Unnamed: 75, 36: Unnamed: 77, 37: Unnamed: 78, 38: Unnamed: 80, 39: Unnamed: 81, 40: Unnamed: 83, 41: Unnamed: 84, 42: Unnamed: 86, 43: Unnamed: 87, 44: Unnamed: 89, 45: Unnamed: 90, 46: Unnamed: 92, 47: Unnamed: 93, 48: Unnamed: 95, 49: Unnamed: 96, 50: Unnamed: 98, 51: Unnamed: 99, 52: Unnamed: 101, 53: Unnamed: 102, 5

In [3]:
# Filtrar apenas dados de Passo Fundo
df_pf = df_crimes[df_crimes['municipio fato'] == 'passo fundo'].copy()

# Criar coluna auxiliar para merge
df_pf['cidade'] = 'passo fundo'

In [4]:
# Agrupar os dados por dia (nível diário para cruzar com meteorologia)
df_diario = (
    df_pf.groupby(['data fato', 'cidade'], as_index=False)
    .agg(
        ocorrencias=('data fato', 'count'),
        vitimas_total=('quantidade vítimas', 'sum'),
        idade_media_vitima=('idade vítima', 'mean'),

        # Contagem de tipos específicos de crime
        furtos=('tipo enquadramento', lambda x: x.str.contains('furto', na=False).sum()),
        roubos=('tipo enquadramento', lambda x: x.str.contains('roubo', na=False).sum()),
        homicidios=('tipo enquadramento', lambda x: x.str.contains('homic', na=False).sum())
    )
)

In [5]:
# Leitura do arquivo meteorológico (pulando metadados iniciais)
df_meteo = pd.read_csv(
    'dados_tempo21-25/pfdados.csv',
    sep=';',
    encoding='latin1',
    skiprows=10
)

# Padronizar nomes das colunas (remover acentos e inconsistências)
df_meteo.columns = (
    df_meteo.columns
    .str.lower()
    .str.strip()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

# Remover colunas vazias ou inúteis
df_meteo = df_meteo.loc[:, ~df_meteo.columns.str.contains('^unnamed')]
if '' in df_meteo.columns:
    df_meteo = df_meteo.drop(columns=[''])

In [6]:
# Criar mapeamento automático de colunas
mapa = {}

for col in df_meteo.columns:
    if 'data' in col:
        mapa[col] = 'data fato'
    elif 'precipitacao' in col:
        mapa[col] = 'precipitacao'
    elif 'temperatura maxima' in col:
        mapa[col] = 'temperatura_maxima'
    elif 'temperatura minima' in col:
        mapa[col] = 'temperatura_minima'
    elif 'umidade relativa' in col:
        mapa[col] = 'umidade_relativa'
    elif 'vento' in col and 'velocidade' in col:
        mapa[col] = 'vento_velocidade'

# Aplicar renomeação
df_meteo = df_meteo.rename(columns=mapa)

In [7]:
# Converter data
df_meteo['data fato'] = pd.to_datetime(df_meteo['data fato'], errors='coerce')

# Converter colunas numéricas (corrigir vírgula decimal)
for col in ['precipitacao', 'temperatura_maxima', 'temperatura_minima', 'umidade_relativa', 'vento_velocidade']:
    if col in df_meteo.columns:
        df_meteo[col] = (
            df_meteo[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df_meteo[col] = pd.to_numeric(df_meteo[col], errors='coerce')

# Criar temperatura média
if 'temperatura_maxima' in df_meteo.columns and 'temperatura_minima' in df_meteo.columns:
    df_meteo['temperatura_media'] = (
        df_meteo['temperatura_maxima'] + df_meteo['temperatura_minima']
    ) / 2

# Criar coluna cidade
df_meteo['cidade'] = 'passo fundo'

# Remover duplicatas
df_meteo = df_meteo.drop_duplicates(subset=['data fato', 'cidade'])

In [8]:
# Merge entre criminalidade e meteorologia
df_integrado = pd.merge(
    df_diario,
    df_meteo[['data fato', 'cidade', 'precipitacao', 'temperatura_maxima',
              'temperatura_minima', 'temperatura_media',
              'umidade_relativa', 'vento_velocidade']],
    on=['data fato', 'cidade'],
    how='left'
)

In [9]:
# Ordenar por data
df_integrado = df_integrado.sort_values('data fato')

colunas_meteo = [
    'precipitacao', 'temperatura_maxima', 'temperatura_minima',
    'temperatura_media', 'umidade_relativa', 'vento_velocidade'
]

# Interpolação temporal (ideal para dados contínuos como clima)
for col in colunas_meteo:
    if col in df_integrado.columns:
        df_integrado[col] = df_integrado[col].interpolate(limit_direction='both')

# Preencher valores restantes com mediana
for col in colunas_meteo:
    if col in df_integrado.columns:
        df_integrado[col] = df_integrado[col].fillna(df_integrado[col].median())

In [10]:
df_integrado.info()
df_integrado.head()

<class 'pandas.DataFrame'>
RangeIndex: 1612 entries, 0 to 1611
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   data fato           1612 non-null   datetime64[us]
 1   cidade              1612 non-null   str           
 2   ocorrencias         1612 non-null   int64         
 3   vitimas_total       1612 non-null   float64       
 4   idade_media_vitima  1612 non-null   float64       
 5   furtos              1612 non-null   int64         
 6   roubos              1612 non-null   int64         
 7   homicidios          1612 non-null   int64         
 8   precipitacao        1612 non-null   float64       
 9   temperatura_maxima  1612 non-null   float64       
 10  temperatura_minima  1612 non-null   float64       
 11  temperatura_media   1612 non-null   float64       
 12  umidade_relativa    1612 non-null   float64       
 13  vento_velocidade    1612 non-null   float64       
dtypes: 

,data fato,cidade,ocorrencias,vitimas_total,idade_media_vitima,furtos,roubos,homicidios,precipitacao,temperatura_maxima,temperatura_minima,temperatura_media,umidade_relativa,vento_velocidade
0,2021-10-01,passo fundo,53,39.0,42.538462,8,0,0,10.6,23.0,15.6,19.30,91.6,3.1
1,2021-10-02,passo fundo,41,37.0,40.250000,3,0,0,0.0,24.0,13.3,18.65,83.2,2.8
2,2021-10-03,passo fundo,36,37.0,36.371429,3,2,2,7.0,20.1,16.2,18.15,92.0,2.1
3,2021-10-04,passo fundo,33,31.0,36.793103,1,1,0,0.8,18.5,13.0,15.75,85.7,2.7
4,2021-10-05,passo fundo,44,41.0,39.527778,5,2,0,0.2,23.1,6.7,14.90,73.4,2.7


In [11]:
# Organizar antes de salvar
df_integrado = df_integrado.sort_values('data fato').reset_index(drop=True)

# Salvar arquivo final
df_integrado.to_csv('etapa4_dados_integrados.csv', index=False)

print("Etapa 4 concluída com sucesso!")

Etapa 4 concluída com sucesso!
